In [19]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Attention, Multiply, Lambda
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l1_l2
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import IsolationForest
import optuna
import warnings
warnings.filterwarnings('ignore')


class SimpleLSTMEarthquakePredictor(BaseEstimator, RegressorMixin):
    
    def __init__(self, sequence_length=10, lstm_units=64, lstm_layers=2,
                 dropout_rate=0.2, dense_dim=32, learning_rate=0.001,
                 batch_size=32, epochs=100, optimizer='adam',
                 activation='relu', random_state=None, use_optuna=False, n_trials=20):
        self.sequence_length = sequence_length
        self.lstm_units = lstm_units
        self.lstm_layers = lstm_layers
        self.dropout_rate = dropout_rate
        self.dense_dim = dense_dim
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.optimizer = optimizer
        self.activation = activation
        self.random_state = random_state
        self.use_optuna = use_optuna
        self.n_trials = n_trials
        self.model = None
        self.feature_scaler = None
        self.target_scaler = None
        self.is_fitted = False
        self.best_params = None
        self.study = None
        
        if self.random_state is not None:
            tf.random.set_seed(self.random_state)
            np.random.seed(self.random_state)
    
    def _create_sliding_windows(self, data):
        X, y = [], []
        for i in range(len(data) - self.sequence_length):
            sequence = data[i:(i + self.sequence_length), :4]
            target = data[i + self.sequence_length, 4:] 
            X.append(sequence)
            y.append(target)
        return np.array(X), np.array(y)
         
    def _prepare_data(self, df):
        feature_cols = ['mag', 'depth', 'latitude', 'longitude']
        target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
        
        available_features = [col for col in feature_cols if col in df.columns]
        available_targets = [col for col in target_cols if col in df.columns]
        
        if len(available_features) != 4 or len(available_targets) != 4:
            raise ValueError(f"Gerekli sütunlar bulunamadı. Mevcut: {df.columns.tolist()}")
        
        all_cols = available_features + available_targets
        data = df[all_cols].values
        data = data[~np.isnan(data).any(axis=1)]
        
        return data
         
    def _build_model(self):#burada yapay zekadan yardım aldığım yer compile ve optimizer kısımlarıydı
        inputs = Input(shape=(self.sequence_length, 4))
        
        x = inputs
        for i in range(self.lstm_layers):
            return_sequences = True if i < self.lstm_layers - 1 else False
            x = LSTM(self.lstm_units, 
                    return_sequences=return_sequences,
                    dropout=self.dropout_rate
                    )(x)
            
            if i < self.lstm_layers - 1:
                x = Dropout(self.dropout_rate)(x)
        
        x = Dense(self.dense_dim, 
                 activation=self.activation,
                 kernel_regularizer=l1_l2(l1=0.01, l2=0.01))(x)
        x = Dropout(self.dropout_rate)(x)
        
        x = Dense(self.dense_dim // 2, 
                 activation=self.activation,
                 kernel_regularizer=l1_l2(l1=0.01, l2=0.01))(x)
        x = Dropout(self.dropout_rate)(x)
        
        outputs = Dense(4, activation='linear')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        
        if self.optimizer == 'adam':
            opt = Adam(learning_rate=self.learning_rate)
        elif self.optimizer == 'rmsprop':
            opt = RMSprop(learning_rate=self.learning_rate)
        else:
            opt = SGD(learning_rate=self.learning_rate)
        
        model.compile(optimizer=opt, 
                     loss='mse',
                     metrics=['mae'])
        
        return model
             
    def _optuna_objective(self, trial, train_data, val_data):
        params = {
            'sequence_length': trial.suggest_int('sequence_length', 5, 20),
            'lstm_units': trial.suggest_categorical('lstm_units', [32, 64, 128, 256]),
            'lstm_layers': trial.suggest_int('lstm_layers', 1, 3),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
            'dense_dim': trial.suggest_categorical('dense_dim', [16, 32, 64, 128]),
            'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-2),
            'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
            'optimizer': trial.suggest_categorical('optimizer', ['adam', 'rmsprop']),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'epochs': 30, 
            'random_state': self.random_state
        }
        
        try:
            temp_model = SimpleLSTMEarthquakePredictor(**params)
            temp_model.fit(train_data)
            
            val_predictions = temp_model.predict(val_data)
           
            if len(val_predictions) == 0:
                return float('inf')
           
            val_actual = val_data[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
            min_len = min(len(val_predictions), len(val_actual))
            
            if min_len == 0:
                return float('inf')
            
            mse = mean_squared_error(
                val_actual.iloc[:min_len, 0],  
                val_predictions[:min_len, 0]  
            )
            
            return mse
            
        except Exception as e:
            print(f"Trial failed: {e}")
            return float('inf')
         
    def _optimize_hyperparameters(self, train_data, val_data):#bayesian search kurulumunu bilmediğim için burada yardım aldım.
        self.study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=self.random_state),
            pruner=optuna.pruners.MedianPruner()
        )
        
        objective_with_data = lambda trial: self._optuna_objective(trial, train_data, val_data)
        self.study.optimize(objective_with_data, n_trials=self.n_trials)
        
        self.best_params = self.study.best_params
        
        # En iyi parametreleri güncel modele uygula
        for key, value in self.best_params.items():
            if hasattr(self, key):
                setattr(self, key, value)
        
        return self.best_params
        
    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Optuna optimizasyonu kullanılacaksa
        if self.use_optuna:
            # Train-validation split için veriyi böl
            train_size = int(0.8 * len(df))
            train_data = df[:train_size]
            val_data = df[train_size:]
            best_params = self._optimize_hyperparameters(train_data, val_data)
            print(f"En iyi parametreler: {best_params}")
            
            # Tüm veriyi kullanarak final model eğitimi
            df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        data = self._prepare_data(df)
        
        self.feature_scaler = MinMaxScaler()
        self.target_scaler = MinMaxScaler()
        
        features_scaled = self.feature_scaler.fit_transform(data[:, :4])
        targets_scaled = self.target_scaler.fit_transform(data[:, 4:])
        data_scaled = np.hstack([features_scaled, targets_scaled])
        
        X_seq, y_seq = self._create_sliding_windows(data_scaled)
        
        if len(X_seq) == 0:
            raise ValueError(f"Yeterli veri yok. Sequence length: {self.sequence_length}, Data length: {len(data_scaled)}")
        
        split_idx = int(0.8 * len(X_seq))
        X_train, X_val = X_seq[:split_idx], X_seq[split_idx:]
        y_train, y_val = y_seq[:split_idx], y_seq[split_idx:]
        
        self.model = self._build_model()
        
        early_stopping = EarlyStopping(#early stop için
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=0
        )
        
        reduce_lr = ReduceLROnPlateau(#learning rate ayarı
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=0
        )
        history = self.model.fit(
            X_train, y_train,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_data=(X_val, y_val),
            callbacks=[early_stopping, reduce_lr],
            verbose=0
        )

       
        self.is_fitted = True
        return self
         
    def predict(self, X):
        if not self.is_fitted:
            raise ValueError("Model henüz eğitilmemiş. Önce fit() metodunu çağırın.")
        
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        data = self._prepare_data(df)
        
        features_scaled = self.feature_scaler.transform(data[:, :4])
        targets_scaled = self.target_scaler.transform(data[:, 4:])
        data_scaled = np.hstack([features_scaled, targets_scaled])
        
        X_seq, _ = self._create_sliding_windows(data_scaled)
        
        if len(X_seq) == 0:
            return np.array([])
        
        predictions_scaled = self.model.predict(X_seq, verbose=0)
        predictions = self.target_scaler.inverse_transform(predictions_scaled)
        
        return predictions




In [20]:
class AdvancedLSTMEarthquakePredictor(BaseEstimator, RegressorMixin):
    
    def __init__(self, sequence_length=10, lstm_units=64, lstm_layers=2, 
                 dropout_rate=0.2, dense_dim=32, learning_rate=0.001,
                 batch_size=32, epochs=100, optimizer='adam', 
                 activation='relu', use_attention=True, random_state=None,
                 use_optuna=False, n_trials=20, use_outlier_detection=True,
                 use_robust_scaling=True):
        self.sequence_length = sequence_length
        self.lstm_units = lstm_units
        self.lstm_layers = lstm_layers
        self.dropout_rate = dropout_rate
        self.dense_dim = dense_dim
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.optimizer = optimizer
        self.activation = activation
        self.use_attention = use_attention
        self.random_state = random_state
        self.use_optuna = use_optuna
        self.n_trials = n_trials
        self.use_outlier_detection = use_outlier_detection
        self.use_robust_scaling = use_robust_scaling
        self.model = None
        self.feature_scaler = None
        self.target_scaler = None
        self.outlier_detector = None
        self.robust_scaler = None
        self.is_fitted = False
        self.best_params = None
        self.study = None
   
        if self.random_state is not None:
            tf.random.set_seed(self.random_state)
            np.random.seed(self.random_state)
         
    def _create_sliding_windows(self, data):
        X, y = [], []
        for i in range(len(data) - self.sequence_length):
            sequence = data[i:(i + self.sequence_length), :4] 
            target = data[i + self.sequence_length, 4:] 
            X.append(sequence)
            y.append(target)
        return np.array(X), np.array(y)
         
    def _preprocess_advanced(self, df, is_training=False):
        df_processed = df.copy()
       
        # Outlier detection
        if self.use_outlier_detection and is_training:
            numerical_cols = ['mag', 'latitude', 'longitude', 'depth']
            available_cols = [col for col in numerical_cols if col in df_processed.columns]
            
            if available_cols:
                self.outlier_detector = IsolationForest(
                    contamination=0.1,
                    random_state=self.random_state,
                    n_estimators=100
                )
                outlier_mask = self.outlier_detector.fit_predict(df_processed[available_cols]) == 1
                df_processed = df_processed[outlier_mask]
                
        
        # Robust scaling
        if self.use_robust_scaling:
            robust_cols = ['mag', 'latitude', 'longitude', 'depth']
            available_robust_cols = [col for col in robust_cols if col in df_processed.columns]
            
            if available_robust_cols:
                if is_training:
                    self.robust_scaler = RobustScaler()
                    robust_scaled_data = self.robust_scaler.fit_transform(df_processed[available_robust_cols])
                else:
                    robust_scaled_data = self.robust_scaler.transform(df_processed[available_robust_cols])
                
                for i, col in enumerate(available_robust_cols):
                    df_processed[f'{col}_robust'] = robust_scaled_data[:, i]
            
        
        return df_processed
         
    def _prepare_data(self, df, is_training=False):
        # Gelişmiş preprocessing uygula
        df_processed = self._preprocess_advanced(df, is_training)
        
        feature_cols = ['mag', 'depth', 'latitude', 'longitude']
        target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
        
        available_features = [col for col in feature_cols if col in df_processed.columns]
        available_targets = [col for col in target_cols if col in df_processed.columns]
        
        if len(available_features) != 4 or len(available_targets) != 4:
            raise ValueError(f"Gerekli sütunlar bulunamadı. Mevcut: {df_processed.columns.tolist()}")
        
        all_cols = available_features + available_targets
        data = df_processed[all_cols].values
        data = data[~np.isnan(data).any(axis=1)]
        
        return data
         
    def _build_model(self):
        inputs = Input(shape=(self.sequence_length, 4))
        
        x = inputs
        for i in range(self.lstm_layers):
            return_sequences = True if i < self.lstm_layers - 1 or self.use_attention else False
            x = LSTM(self.lstm_units, 
                    return_sequences=return_sequences,
                    dropout=self.dropout_rate
                    )(x)
            
            if i < self.lstm_layers - 1:
                x = Dropout(self.dropout_rate)(x)
        
        # Attention mekanizması,burada yapay zeka ile  beraber mekanizmayı kurarken Kullandığım 
        # Lambda ların keeras dan izin istemesi üzerine output shape eklemem gerekti
        if self.use_attention:
            attention_weights = Dense(1, activation='tanh')(x)
            attention_weights = Lambda(lambda x: tf.nn.softmax(x, axis=1), output_shape=lambda s: s)(attention_weights)
            context_vector = Multiply()([x, attention_weights])
            context_vector = Lambda(lambda x: tf.reduce_sum(x, axis=1),output_shape=lambda s: (s[0], s[2]))(context_vector)
            x = context_vector
        
        x = Dense(self.dense_dim, 
                 activation=self.activation,
                 kernel_regularizer=l1_l2(l1=0.01, l2=0.01))(x)
        x = Dropout(self.dropout_rate)(x)
        
        x = Dense(self.dense_dim // 2, 
                 activation=self.activation,
                 kernel_regularizer=l1_l2(l1=0.01, l2=0.01))(x)
        x = Dropout(self.dropout_rate)(x)
        
        outputs = Dense(4, activation='linear')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        
        if self.optimizer == 'adam':
            opt = Adam(learning_rate=self.learning_rate)
        elif self.optimizer == 'rmsprop':#rmsprop RNN ile çok iyi çalışır,gradyan inişte ağırlıkları
            opt = RMSprop(learning_rate=self.learning_rate)
        else:
            opt = SGD(learning_rate=self.learning_rate)
        
        model.compile(optimizer=opt, 
                     loss='mse',
                     metrics=['mae'])
        
        return model
      #optimizer ı kurarken yapay zeka yardımı aldım   
    def _optuna_objective(self, trial, train_data, val_data):
        params = {
            'sequence_length': trial.suggest_int('sequence_length', 5, 20),
            'lstm_units': trial.suggest_categorical('lstm_units', [32, 64, 128, 256]),
            'lstm_layers': trial.suggest_int('lstm_layers', 1, 3),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
            'dense_dim': trial.suggest_categorical('dense_dim', [16, 32, 64, 128]),
            'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-2),
            'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
            'optimizer': trial.suggest_categorical('optimizer', ['adam', 'rmsprop']),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'epochs': 30,
            'random_state': self.random_state,
            'use_outlier_detection': self.use_outlier_detection,
            'use_robust_scaling': self.use_robust_scaling
        }
        
        try:
            temp_model = AdvancedLSTMEarthquakePredictor(**params)
            temp_model.fit(train_data)
            
            val_predictions = temp_model.predict(val_data)
            
            if len(val_predictions) == 0:
                return float('inf')
            
            val_actual = val_data[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
            min_len = min(len(val_predictions), len(val_actual))
            
            if min_len == 0:
                return float('inf')
            
            mse = mean_squared_error(
                val_actual.iloc[:min_len, 0],  
                val_predictions[:min_len, 0]  
            )
            
            return mse
            
        except Exception as e:
            print(f"Trial failed: {e}")
            return float('inf')
         
    def _optimize_hyperparameters(self, train_data, val_data):
        self.study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=self.random_state),
            pruner=optuna.pruners.MedianPruner()
        )
        
        objective_with_data = lambda trial: self._optuna_objective(trial, train_data, val_data)
        self.study.optimize(objective_with_data, n_trials=self.n_trials)
        
        self.best_params = self.study.best_params
        
        # En iyi parametreleri güncel modele uygula
        for key, value in self.best_params.items():
            if hasattr(self, key):
                setattr(self, key, value)
        
        return self.best_params
         
    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
       
        if self.use_optuna:
            train_size = int(0.8 * len(df))
            train_data = df[:train_size]
            val_data = df[train_size:]
            best_params = self._optimize_hyperparameters(train_data, val_data)
            print(f"En iyi parametreler: {best_params}")
            
            df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        
        data = self._prepare_data(df, is_training=True)
        
        self.feature_scaler = MinMaxScaler()
        self.target_scaler = MinMaxScaler()
        
        features_scaled = self.feature_scaler.fit_transform(data[:, :4])
        targets_scaled = self.target_scaler.fit_transform(data[:, 4:])
        data_scaled = np.hstack([features_scaled, targets_scaled])
        
        
        
        X_seq, y_seq = self._create_sliding_windows(data_scaled)
        
        if len(X_seq) == 0:
            raise ValueError(f"Yeterli veri yok. Sequence length: {self.sequence_length}, Data length: {len(data_scaled)}")
        
        split_idx = int(0.8 * len(X_seq))
        X_train, X_val = X_seq[:split_idx], X_seq[split_idx:]
        y_train, y_val = y_seq[:split_idx], y_seq[split_idx:]
        
        self.model = self._build_model()
        
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=0
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=0
        )
        
        history = self.model.fit(
            X_train, y_train,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_data=(X_val, y_val),
            callbacks=[early_stopping, reduce_lr],
            verbose=0
        )
        
        self.is_fitted = True
        return self
         
    def predict(self, X):
        if not self.is_fitted:
            raise ValueError("Model henüz eğitilmemiş. Önce fit() metodunu çağırın.")
        
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        data = self._prepare_data(df, is_training=False)
        
        features_scaled = self.feature_scaler.transform(data[:, :4])
        targets_scaled = self.target_scaler.transform(data[:, 4:])
        data_scaled = np.hstack([features_scaled, targets_scaled])
        
        X_seq, _ = self._create_sliding_windows(data_scaled)
        
        if len(X_seq) == 0:
            return np.array([])
        
        predictions_scaled = self.model.predict(X_seq, verbose=0)
        predictions = self.target_scaler.inverse_transform(predictions_scaled)
        
        return predictions
